# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'brand/partner page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discourse forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints', 'url': 'https://endpoi

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-4.7-Flash
Updated
6 days ago
•
450k
•
1.2k
nvidia/personaplex-7b-v1
Updated
3 days ago
•
34.9k
•
1.02k
microsoft/VibeVoice-ASR
Updated
5 days ago
•
46.9k
•
539
Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice
Updated
3 days ago
•
91.8k
•
505
unsloth/GLM-4.7-Flash-GGUF
Updated
2 days ago
•
228k
•
332
Browse 2M+ models
Spaces
Running
on
Zero
Featured
702
Qwen3-TTS Demo
🎙
702
Convert text to speech with custom voices and cloning
Running
on
Zero
Featured
1.16k
Qwen Image Multiple Angles 3D Camera
🎥
1.16k
Adjust camera angles in images usi

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-4.7-Flash\nUpdated\n6 days ago\n•\n450k\n•\n1.2k\nnvidia/personaplex-7b-v1\nUpdated\n3 days ago\n•\n34.9k\n•\n1.02k\nmicrosoft/VibeVoice-ASR\nUpdated\n5 days ago\n•\n46.9k\n•\n539\nQwen/Qwen3-TTS-12Hz-1.7B-CustomVoice\nUpdated\n3 days ago\n•\n91.8k\n•\n505\nunsloth/GLM-4.7-Flash-GGUF\nUpdated\n2 days ago\n•\n228k\n•\n332\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n7

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the global AI community and collaboration platform dedicated to building the future of machine learning (ML). Serving as a central hub, it empowers ML engineers, scientists, and end users worldwide to create, share, and explore open-source ML models, datasets, and applications. Together, the community is shaping an open and ethical AI future.

- **Mission:** To build an open and ethical AI future through collaboration.
- **Platform:** Home to over 2 million ML models, 500,000+ datasets, and 1 million+ applications.
- **Specialties:** Supporting all AI modalities including text, image, video, audio, and 3D.
- **Open Source:** The Hugging Face Hub is the go-to place for discovering and experimenting with open-source machine learning resources.
- **Community:** A fast-growing ecosystem with a talented science team, fostering knowledge sharing and innovation in AI.

---

## What We Offer

### Models  
Explore over 2 million machine learning models, including trending models like GLM-4.7-Flash, Personaplex-7b-v1, VibeVoice-ASR, and Qwen3-TTS-12Hz. These models cover a wide range of AI domains, from speech recognition and text-to-speech to image generation and beyond.

### Datasets  
Access and contribute to a large collection of over 500,000 datasets, with frequent updates supporting superior reasoning, OCR, and entertainment data, enabling researchers and developers to train better models.

### Spaces  
Experiment with and build interactive AI apps on Hugging Face’s Spaces — a place to run, host, and demo ML-powered applications. Featured Spaces include demos converting text to speech, image editing tools, and 3D image angle adjustments.

### Enterprise Solutions  
Tailored products and services for enterprise customers to leverage Hugging Face’s technology stack in commercial applications.

---

## Company Culture

Hugging Face thrives as a community-first organization deeply rooted in openness, collaboration, and innovation. The company cultivates an environment that empowers the next generation of AI professionals by encouraging:

- **Collaboration:** Building tools and platforms that enable sharing and co-creation across the global AI community.
- **Transparency:** Fostering an open-source approach to build trustworthy and ethical AI systems.
- **Innovation:** Pushing the boundaries of what AI can achieve with a talented and diverse team at the technological frontier.
- **Learning & Growth:** Supporting users and employees in building their ML portfolios through public sharing and experimentation.

---

## Career Opportunities

Join Hugging Face in revolutionizing machine learning! The company is actively hiring talented individuals passionate about AI and open source innovation. Current job openings span various roles that contribute to building and supporting the technology powering the ML community.

- Visit the **Careers** page to explore available positions.
- Be part of a collaborative, fast-paced, and mission-driven environment.
- Grow your skills working alongside some of the brightest minds in AI.

---

## Why Choose Hugging Face?

- **Community-Driven:** The pulse of the global ML community.
- **Cutting-Edge Tech:** Access to the most extensive collection of open-source ML models and datasets.
- **Multi-Modal ML:** Support for text, image, video, audio, and 3D AI applications.
- **Ethical AI:** Commitment to transparency and building responsible AI technologies.
- **User Empowerment:** Tools designed to help users showcase their work and grow professionally.

---

## Connect with Hugging Face

- Website: [huggingface.co](https://huggingface.co)
- GitHub, Twitter, LinkedIn, Discord (active social presence)
- For assets and brand information, visit the Brand Assets page for logos and color guidelines.

Build the future of AI with Hugging Face — where collaboration creates innovation.

---

*Hugging Face — The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face - The AI Community Building the Future

---

## Overview

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. It serves as a central hub where engineers, scientists, researchers, and developers can share, explore, and experiment with open-source machine learning models, datasets, and applications. With a fast-growing community and one of the largest collections of ML resources, Hugging Face empowers the next generation of AI practitioners to build an open, ethical, and innovative AI future.

Explore over 2 million models, 500k+ datasets, and 1 million+ AI applications across multiple modalities including text, image, video, audio, and even 3D. From cutting-edge model hosting to interactive demos (called Spaces), Hugging Face connects users worldwide to accelerate AI development and deployment.

---

## Company Culture & Values

- **Open & Collaborative:** Hugging Face fosters a vibrant open-source community where sharing code, datasets, and ideas is encouraged.
- **Innovative & Inclusive:** The team values creativity and inclusivity, aiming to empower a broad range of users from beginners to advanced AI researchers.
- **Ethical AI:** Committed to building transparent and responsible AI technologies that benefit society.
- **Learning & Growth:** The platform supports users in building their portfolios and deepening their expertise through collaboration and experimentation.

---

## Key Offerings

### Hugging Face Hub
- A one-stop repository hosting 2M+ open-source ML models.
- Easy browsing of datasets and applications contributed by the community.
- Tools to share and build your ML profile and portfolio.

### Spaces
- Interactive ML apps and demos that run in the cloud.
- Supports text-to-speech, image generation, 3D control, and more.
- Allows rapid prototype sharing and user engagement.

### Enterprise Solutions
- Scalable AI platform tailored for organizations.
- Features include:
  - Enterprise-grade security and access controls
  - Single sign-on (SSO) integration
  - Detailed audit logs and analytics dashboards
  - Enhanced compute options and quota boosts
  - Private storage and dataset viewers for confidential collaboration
- Flexible subscription plans starting at $20/user/month with custom enterprise contracts.

---

## Customers & Community

Hugging Face supports a wide range of users, including:

- AI startups and tech enterprises looking to build state-of-the-art solutions.
- Academic institutions and researchers advancing AI knowledge.
- Individual developers and hobbyists eager to experiment with machine learning.
- Large organizations requiring secure and scalable AI infrastructures.

The platform’s extensive, active user base benefits from continuously updated models and datasets, fostering strong peer collaboration and innovation.

---

## Careers at Hugging Face

Join a passionate global team dedicated to shaping the future of AI. Hugging Face offers opportunities for:

- Machine Learning Engineers & Researchers
- Software Developers
- Community Managers & Advocates
- Data Scientists
- Enterprise Solutions Experts

Working at Hugging Face means engaging with bleeding-edge ML technologies in an open, inclusive environment with a mission to democratize AI.

---

## Connect & Learn More

- **Website:** https://huggingface.co
- **Community Forum:** Engage with users and experts
- **GitHub:** Explore open-source libraries and tools
- **Social:** Follow on Twitter, LinkedIn, Discord for updates
- **Documentation & Blog:** Learn from tutorials, case studies, and news

---

## Brand Identity

- Signature colors: Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)
- Friendly and approachable brand symbolizing openness and innovation in AI

---

Hugging Face — Where the machine learning community comes together to build the future. Join today to contribute, learn, and grow with the leaders of the AI revolution!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>